# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Amankeldy Abylaj
**Student ID:** _(fill in)_
**Group:** _(fill in)_
**Repository:** https://github.com/abylajamankeldy/css4007-hw3

## AI tool disclosure

- **Claude (Anthropic)** helped me write the code cells, set up the notebook and work out Part A step by step. Part A arithmetic was also re-checked by code (the check cell below).
- **Model in Part C:** I had no OpenAI key, so every Part C call uses **`openai/gpt-oss-120b` through Groq's OpenAI-compatible API** (the cell that creates the client prints the provider and model name). The README names `gpt-5.6-luna`; the three systems still use the same model, so the comparison between them is fair, but the absolute answers may differ from what `gpt-5.6-luna` would give.
- The written analysis in Parts B and C and the final questions quotes the outputs printed by this notebook on my own run.
- No API key is in this notebook or in the repository (`.env` is gitignored).

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large · language · models · can · generate · text · and · answer · questions | 9 |
| D2 | retrieval · augmented · generation · retrieves · relevant · documents · before · generating · an · answer | 10 |
| D3 | embeddings · represent · text · as · numerical · vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**

**Query terms:** `how`, `does`, `rag`, `retrieve`, `information`  (the query lowercased, no stop words removed, no stemming)

**TF, DF and IDF of each query term** (N = 3). The idf formula is `ln((N − n + 0.5) / (n + 0.5))`:

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

- `how`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `does`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `rag`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `retrieve`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `information`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**

Every query term appears in **zero** documents. The word `retrieve` is not `retrieves` and not `retrieval` — without stemming these are three different tokens. `rag` is never written out in D2 (only the words "Retrieval-Augmented Generation" are), and `does`, `how`, `information` occur nowhere.

**BM25 of each document** (a term contributes only if f(t, d) > 0, because the numerator `idf · f · (k1+1)` is then 0):

> - **D1:** `how` f=0, `does` f=0, `rag` f=0, `retrieve` f=0, `information` f=0 → score = 0 + 0 + 0 + 0 + 0 = **0**
> - **D2:** `how` f=0, `does` f=0, `rag` f=0, `retrieve` f=0 (D2 has `retrieves` and `retrieval`, not `retrieve`), `information` f=0 → score = **0**
> - **D3:** all five f=0 → score = **0**

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

The three-way tie at zero is not an arithmetic mistake. I rechecked: no query token is equal to any document token, so BM25 has nothing to rank by. BM25 cannot pick a first document here.

## A1 (b) — BM25 with the stems from README §3

Stems used: retrieve / retrieves / retrieval → `retriev`; generate / generating / generation → `gener`; information → `inform`; does → `doe`. Every other word stays as it is.

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large · language · models · can · gener · text · and · answer · questions | 9 |
| D2 | retriev · augmented · gener · retriev · relevant · documents · before · gener · an · answer | 10 |
| D3 | embeddings · represent · text · as · numerical · vectors | 6 |

**avgdl** = (9 + 10 + 6) / 3 = 25 / 3 = **8.3333**  (stemming does not change the number of tokens)

**Stemmed query terms:** `how`, `doe`, `rag`, `retriev`, `inform`

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

- `how`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `doe`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `rag`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**
- `retriev`: n(t) = 1 → idf = ln((3 − 1 + 0.5) / (1 + 0.5)) = ln(2.5 / 1.5) = ln(1.6667) = **0.5108**
- `inform`: n(t) = 0 → idf = ln((3 − 0 + 0.5) / (0 + 0.5)) = ln(3.5 / 0.5) = ln(7.0000) = **1.9459**

Only `retriev` now matches, and only in D2 (twice: `retrieval` and `retrieves`). D1 and D3 contain no query term (D1 has `gener`, which is not a query term).

**BM25 of each document**, worked:

> **D1** = 0 (none of `how`, `doe`, `rag`, `retriev`, `inform` occurs in D1)
>
> **D2** = only the term `retriev` contributes:
>
> - **D2**, term `retriev`: idf = 0.5108, f = 2, |d| = 10, avgdl = 8.3333
>   - length part: 1 − b + b·|d|/avgdl = 1 − 0.75 + 0.75·10/8.3333 = 1.1500
>   - denominator: f + k1·(that) = 2 + 1.5·1.1500 = 3.7250
>   - numerator: idf·f·(k1+1) = 0.5108·2·2.5 = 2.5541
>   - term score = 2.5541 / 3.7250 = **0.6857**
>
> **D3** = 0 (no query term occurs in D3)

| Document | BM25 Score | Rank |
|---|---:|---:|
| D1 | 0.0000 | 2 (tie) |
| D2 | 0.6857 | 1 |
| D3 | 0.0000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

> - **(a):** none. All three documents score 0, so BM25 has no first document. The query asks about "retrieve", but D2 says "retrieves" and "retrieval", and without stemming a token has to be identical to count. The result looked like a mistake, I checked every term again, and it survives the check, so it is the real behaviour of exact-token matching.
> - **(b):** **D2**, score 0.6857. After stemming, `retrieves` and `retrieval` both become `retriev`, so the query term `retriev` is found twice in D2. D1 and D3 still score 0.
> - The only thing that changed between (a) and (b) is the tokenisation, not the documents or the formula. D2 is also the document a human would pick, but BM25 got there only through shared word stems, never through meaning. Note that `rag`, `how`, `doe` and `inform` still match nothing: D2 never writes "RAG" or "information".

## A2 — cosine similarity

```
Query = [0.8, 0.2, 0.5]
D1 = [0.7, 0.1, 0.4]   D2 = [0.9, 0.3, 0.6]   D3 = [0.1, 0.8, 0.2]
```

Formula: cosine(Q, D) = (Q · D) / (‖Q‖ · ‖D‖), with Q · D = Σ qᵢ·dᵢ and ‖v‖ = √(Σ vᵢ²).

- ‖Query‖ = √(0.8² + 0.2² + 0.5²) = √(0.64 + 0.04 + 0.25) = √0.93 = 0.9644
- ‖D1‖ = √(0.7² + 0.1² + 0.4²) = √(0.49 + 0.01 + 0.16) = √0.66 = 0.8124
- ‖D2‖ = √(0.9² + 0.3² + 0.6²) = √(0.81 + 0.09 + 0.36) = √1.26 = 1.1225
- ‖D3‖ = √(0.1² + 0.8² + 0.2²) = √(0.01 + 0.64 + 0.04) = √0.69 = 0.8307
- cos(Query, D1) = 0.78 / (0.9644 · 0.8124) = 0.78 / 0.7835 = **0.9956**
- cos(Query, D2) = 1.08 / (0.9644 · 1.1225) = 1.08 / 1.0825 = **0.9977**
- cos(Query, D3) = 0.34 / (0.9644 · 0.8307) = 0.34 / 0.8011 = **0.4244**

| Pair | Dot product (worked) | ‖Query‖ | ‖D‖ | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.8·0.7 + 0.2·0.1 + 0.5·0.4 = 0.56 + 0.02 + 0.20 = 0.78 | 0.9644 | 0.8124 | 0.9956 |
| Query · D2 | 0.8·0.9 + 0.2·0.3 + 0.5·0.6 = 0.72 + 0.06 + 0.30 = 1.08 | 0.9644 | 1.1225 | 0.9977 |
| Query · D3 | 0.8·0.1 + 0.2·0.8 + 0.5·0.2 = 0.08 + 0.16 + 0.10 = 0.34 | 0.9644 | 0.8307 | 0.4244 |

| Document | Cosine Similarity | Rank |
|---|---:|---:|
| D1 | 0.9956 | 2 |
| D2 | 0.9977 | 1 |
| D3 | 0.4244 | 3 |

**Which document is most similar to the query?**

> **D2** (cosine 0.9977). D1 is very close behind (0.9956), and D3 is far away (0.4244): D3's vector points mostly along the second axis, the query's mostly along the first.

**Is the ranking the same as BM25's?** Compare it with both A1 rankings.

> The cosine ranking is **D2 > D1 > D3**.
> - Against **A1 (a)**: not comparable — BM25 gave a three-way tie at 0, so it said nothing; cosine produces a full ranking.
> - Against **A1 (b)**: the **top document is the same (D2)**. The rest differs: BM25 (b) leaves D1 and D3 tied at 0, whereas cosine puts D1 clearly second (0.9956) and D3 last (0.4244). So the two methods agree on the winner after stemming, and cosine also tells how close D1 is.

**Why can BM25 and embedding retrieval disagree?** Point at the words of this query and these documents.

> BM25 counts identical tokens. The query words `retrieve` and `information` are not in any document as written, so in (a) BM25 sees no overlap at all; even in (b) it only finds `retriev` in D2 and never sees that D1 ("answer questions") is also about the same topic. An embedding model compares meaning: "How does RAG retrieve information?" is close to "Retrieval-Augmented Generation retrieves relevant documents…" because RAG *is* retrieval-augmented generation, a fact BM25 cannot know from the letters `rag`. It is also close to D1 (language models that answer questions), because both are about question answering, which shares no words with the query. The weakness is the opposite one: embeddings blur exact strings (a form code such as GR-305), where BM25 is exact.
> (Here the vectors were given, so the "meaning" is already baked into the numbers. D1 and D2 are both near the query because their vectors point in nearly the same direction.)

Check of the Part A arithmetic (the working above is written out by hand; this cell only confirms the numbers):

In [1]:
import math
k1, b = 1.5, 0.75
D = {"D1": "large language models can generate text and answer questions".split(),
     "D2": "retrieval augmented generation retrieves relevant documents before generating an answer".split(),
     "D3": "embeddings represent text as numerical vectors".split()}
STEM = {"retrieve": "retriev", "retrieves": "retriev", "retrieval": "retriev", "generate": "gener",
        "generating": "gener", "generation": "gener", "information": "inform", "does": "doe"}
query = "how does rag retrieve information".split()

def bm25_by_hand(docs, q):
    N = len(docs); avg = sum(len(w) for w in docs.values()) / N
    scores = {}
    for d, w in docs.items():
        s = 0
        for t in q:
            n = sum(1 for ww in docs.values() if t in ww)
            idf = math.log((N - n + 0.5) / (n + 0.5))
            f = w.count(t)
            s += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(w) / avg))
        scores[d] = round(s, 4)
    return scores

print("(a) as written :", bm25_by_hand(D, query))
Ds = {d: [STEM.get(x, x) for x in w] for d, w in D.items()}
print("(b) with stems :", bm25_by_hand(Ds, [STEM.get(x, x) for x in query]))

Q = [0.8, 0.2, 0.5]
V = {"D1": [0.7, 0.1, 0.4], "D2": [0.9, 0.3, 0.6], "D3": [0.1, 0.8, 0.2]}
norm = lambda v: math.sqrt(sum(x * x for x in v))
for d, v in V.items():
    dot = sum(x * y for x, y in zip(Q, v))
    print(d, "dot =", round(dot, 2), "|Q| =", round(norm(Q), 4), "|D| =", round(norm(v), 4), "cos =", round(dot / (norm(Q) * norm(v)), 4))

(a) as written : {'D1': 0.0, 'D2': 0.0, 'D3': 0.0}
(b) with stems : {'D1': 0.0, 'D2': 0.6857, 'D3': 0.0}
D1 dot = 0.78 |Q| = 0.9644 |D| = 0.8124 cos = 0.9956
D2 dot = 1.08 |Q| = 0.9644 |D| = 1.1225 cos = 0.9977
D3 dot = 0.34 |Q| = 0.9644 |D| = 0.8307 cos = 0.4244


---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [2]:
import json, re, os, math
import numpy as np
from dotenv import load_dotenv
load_dotenv()          # reads OPENAI_API_KEY / GROQ_API_KEY / HF_TOKEN from .env (never printed)

corpus  = [json.loads(l) for l in open("data/corpus.jsonl", encoding="utf-8")]
queries = json.load(open("data/queries.json", encoding="utf-8"))["queries"]
qa      = json.load(open("data/qa.json", encoding="utf-8"))["questions"]
ids     = [c["id"] for c in corpus]
by_id   = {c["id"]: c for c in corpus}
print(len(corpus), "chunks,", len(queries), "fixed queries,", len(qa), "questions")

74 chunks, 8 fixed queries, 6 questions


## B1 — BM25 with `rank_bm25`

**My tokenisation:** lowercase, then `re.findall(r"\w+", text)` (a token is a run of letters or digits; the regex is Unicode-aware, so Kazakh and Russian words are kept whole). **No stemming and no stop-word removal**, so `retrieve` and `retrieves` stay different tokens, as in A1 (a). A form code such as `GR-305` becomes two tokens, `gr` and `305`, because the hyphen splits it. `BM25Okapi(k1=1.5, b=0.75)`.

In [3]:
from rank_bm25 import BM25Okapi

def tok(s):
    return re.findall(r"\w+", s.lower())

bm25 = BM25Okapi([tok(c["text"]) for c in corpus], k1=1.5, b=0.75)

def search_bm25(q, k=3):
    s = bm25.get_scores(tok(q))
    top = np.argsort(-s, kind="stable")[:k]
    return [(ids[i], float(s[i])) for i in top]

def first_line(cid, n=80):
    c = by_id[cid]
    return (c["title"] + " | " + c["text"].split("\n")[0])[:n]

print("example:", search_bm25(queries[0]["text"]))

example: [('GO-08', 15.825491699499644), ('GO-12', 8.195256931904641), ('GO-06', 8.07003113015204)]


## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [4]:
from sentence_transformers import SentenceTransformer
import os
import numpy as np

model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=os.environ.get("HF_TOKEN")
)

doc_emb = model.encode_document(
    [c["text"] for c in corpus],
    show_progress_bar=False
)

print("document embeddings:", doc_emb.shape, doc_emb.dtype)

def search_emb(q, k=3):
    qe = model.encode_query(q)
    s = np.asarray(model.similarity(qe, doc_emb)[0])
    top = np.argsort(-s, kind="stable")[:k]
    return [(ids[i], float(s[i])) for i in top]

print("example:", search_emb(queries[0]["text"]))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

document embeddings: (74, 768) float32
example: [('GO-08', 0.6955164074897766), ('GO-12', 0.4608723521232605), ('GO-23', 0.45048606395721436)]


## My own queries

Written **before** running any retriever. `relevant` is decided from reading the handbook, not from results.

| Id | Query | `relevant` | Expected winner, and why |
|---|---|---|---|
| Q-09 | Which office is in room 214? | GO-21 | **BM25.** The query is built around the exact token `214` (and `room`), which appears in only one chunk. An embedding model turns numbers into a blur, and "office" could pull any chunk about the Grant Office. |
| Q-10 | Какие документы подтверждают доход семьи? | GO-10 | **Embeddings.** The question is in Russian ("which documents confirm the family's income") while the answer chunk GO-10 ("Proof of income … salary certificate …") is in English. BM25 shares no words across the two languages; a multilingual embedding model should still match the meaning. |

In [5]:
own = [
    {"id": "Q-09", "text": "Which office is in room 214?",                "relevant": ["GO-21"]},
    {"id": "Q-10", "text": "Какие документы подтверждают доход семьи?",    "relevant": ["GO-10"]},
]
allq = queries + own
print(len(allq), "queries in total")

10 queries in total


## Results for all ten or more queries: top 3 from each retriever

In [6]:
def hit(res, relevant):
    return any(i in relevant for i, _ in res)

results = {}
for q in allq:
    b, e = search_bm25(q["text"]), search_emb(q["text"])
    results[q["id"]] = {"bm25": b, "emb": e, "hb": hit(b, q["relevant"]), "he": hit(e, q["relevant"])}
    print(f"{q['id']}  {q['text']}")
    print(f"   relevant: {q['relevant']}")
    for name, res, h in (("BM25", b, results[q['id']]['hb']), ("EmbeddingGemma", e, results[q['id']]['he'])):
        print(f"   {name}   hit@3 = {'YES' if h else 'no'}")
        for r, (cid, sc) in enumerate(res, 1):
            mark = "*" if cid in q["relevant"] else " "
            print(f"     {r}. {cid}{mark} {sc:8.4f}  {first_line(cid)}")
    print()
print("(* = chunk is in `relevant`)")

Q-01  What is form GR-305 used for?
   relevant: ['GO-08', 'GO-12', 'GO-23']
   BM25   hit@3 = YES
     1. GO-08*  15.8255  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Off
     2. GO-12*   8.1953  How to appeal a decision | A student who disagrees with a decision may appeal wi
     3. GO-06    8.0700  Form GR-104: the application form | Form GR-104 is the application form for the 
   EmbeddingGemma   hit@3 = YES
     1. GO-08*   0.6955  Form GR-305: appeals | Form GR-305 is used to appeal a decision of the Grant Off
     2. GO-12*   0.4609  How to appeal a decision | A student who disagrees with a decision may appeal wi
     3. GO-23*   0.4505  Апелляция: кратко | Если студент не согласен с решением, он может подать апелляц

Q-02  My family is poor. Is there financial help for students like me?
   relevant: ['GO-01', 'GO-02', 'GO-03']
   BM25   hit@3 = no
     1. KZ-17    8.7262  Lake Balkhash (part 2) | The lake currently covers about 16,400 km2 (6,300

## B3 — comparison

The table below is printed by the next cell from the same `results` dictionary as the top-3 lists above. The three examples are written after it.

In [7]:
from IPython.display import Markdown, display
rows = ["| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |", "|---|---|:---:|---|:---:|"]
for q in allq:
    r = results[q["id"]]
    rows.append(f"| {q['id']} | {', '.join(i for i, _ in r['bm25'])} | {'✓' if r['hb'] else '✗'} | {', '.join(i for i, _ in r['emb'])} | {'✓' if r['he'] else '✗'} |")
tb = sum(r["hb"] for r in results.values()); te = sum(r["he"] for r in results.values())
rows.append(f"| **hit@3 total** | | **{tb}/{len(allq)}** | | **{te}/{len(allq)}** |")
display(Markdown("\n".join(rows)))

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, KZ-01, KZ-02 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, KZ-06, KZ-11 | ✓ | KZ-05, KZ-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-21, GO-08, GO-07 | ✓ | GO-21, GO-07, KZ-37 | ✓ |
| Q-10 | KZ-01, KZ-02, KZ-03 | ✗ | GO-10, GO-07, GO-13 | ✓ |
| **hit@3 total** | | **6/10** | | **10/10** |

---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [8]:

import os
import time
import tiktoken
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

enc = tiktoken.get_encoding("o200k_base")

def ntok(s):
    return len(enc.encode(s))

if os.getenv("GROQ_API_KEY"):
    client = OpenAI(
        api_key=os.environ["GROQ_API_KEY"],
        base_url="https://api.groq.com/openai/v1"
    )
    MODEL = "openai/gpt-oss-120b"
    GROQ = True
elif os.getenv("OPENAI_API_KEY"):
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    MODEL = "gpt-5.6-luna"
    GROQ = False
else:
    raise SystemExit("Укажи GROQ_API_KEY или OPENAI_API_KEY в .env")

print(
    "provider:", "Groq" if GROQ else "OpenAI",
    "| model:", MODEL
)

_last = [0.0]

def ask(messages):
    """one chat call -> (answer, prompt_tokens, completion_tokens)"""
    kw = {"max_tokens": 4000, "temperature": 0} if GROQ else {}

    for attempt in range(6):
        wait = 2.5 - (time.time() - _last[0])
        if GROQ and wait > 0:
            time.sleep(wait)

        _last[0] = time.time()

        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                **kw
            )
            return (
                (r.choices[0].message.content or "").strip(),
                r.usage.prompt_tokens,
                r.usage.completion_tokens
            )
        except Exception as ex:
            if attempt == 5:
                raise
            print("Retry after error:", type(ex).__name__)
            time.sleep(5 * (attempt + 1))

SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

def context_block(chunk_ids):
    return "\n\n".join(
        f"[{n}] ({cid} | {by_id[cid]['title']})\n{by_id[cid]['text']}"
        for n, cid in enumerate(chunk_ids, 1)
    )

def rag_messages(question, chunk_ids):
    user = (
        "Context:\n" + context_block(chunk_ids)
        + "\n\nQuestion: " + question
    )
    return [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": user}
    ]


provider: Groq | model: openai/gpt-oss-120b


## C1 — the complete prompt, printed before it is sent

In [9]:
q1 = next(x for x in qa if x["id"] == "C-02")
top = search_emb(q1["question"], 3)
c1_ids = [cid for cid, _ in top]
print("EmbeddingGemma top 3:", top, "\n")

msgs = rag_messages(q1["question"], c1_ids)
print("=" * 30, "WHAT THE MODEL RECEIVES", "=" * 30)
for m in msgs:
    print(f"--- role: {m['role']} ---")
    print(m["content"])
    print()
print("=" * 85)

# token accounting with tiktoken o200k_base
sys_t  = ntok(SYSTEM)
q_t    = ntok("Question: " + q1["question"])
ctx_text = "Context:\n" + context_block(c1_ids)
ctx_t  = ntok(ctx_text)
total_parts = sys_t + q_t + ctx_t
whole = ntok(msgs[0]["content"]) + ntok(msgs[1]["content"])

answer, prompt_tokens, completion_tokens = ask(msgs)
print("answer:", answer, "\n")

from IPython.display import Markdown, display
display(Markdown(f"""| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | {sys_t} |
| Question (with the label "Question:") | {q_t} |
| Retrieved context (with the label "Context:" and the block headers) | {ctx_t} |
| **Total input** (sum of the three) | **{total_parts}** |
| **`usage.prompt_tokens` from the response** | **{prompt_tokens}** |"""))
print("tiktoken on the two message strings as a whole:", whole)
print("difference usage - tiktoken sum:", prompt_tokens - total_parts)
per_chunk = [ntok(by_id[c]["text"]) for c in c1_ids]
print("number of retrieved chunks:", len(c1_ids))
print("tokens per chunk (text only):", dict(zip(c1_ids, per_chunk)))
print("total context size:", ctx_t, "tokens,", len(ctx_text), "characters  (chunk text only:", sum(per_chunk), "tokens,", sum(len(by_id[c]["text"]) for c in c1_ids), "characters)")

EmbeddingGemma top 3: [('GO-08', 0.6028919219970703), ('GO-12', 0.5590736269950867), ('GO-11', 0.5342729091644287)] 

============================== WHAT THE MODEL RECEIVES ==============================
--- role: system ---
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

--- role: user ---
Context:
[1] (GO-08 | Form GR-305: appeals)
Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

[2] (GO-12 | How to appeal a decision)
A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staf

| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question (with the label "Question:") | 23 |
| Retrieved context (with the label "Context:" and the block headers) | 279 |
| **Total input** (sum of the three) | **346** |
| **`usage.prompt_tokens` from the response** | **420** |

tiktoken on the two message strings as a whole: 346
difference usage - tiktoken sum: 74
number of retrieved chunks: 3
tokens per chunk (text only): {'GO-08': 68, 'GO-12': 88, 'GO-11': 78}
total context size: 279 tokens, 1278 characters  (chunk text only: 234 tokens, 1150 characters)


The table above is printed by the cell above it (from `ntok(...)` and `response.usage`), not typed. The numbers for the discussion are quoted in the answers below.

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

>

**What exactly does retrieval add to the model's input?**

>

## C2 — three systems, six questions

In [10]:
runs = {}
for q in qa:
    qid = q["id"]
    b_ids = [cid for cid, _ in search_bm25(q["question"], 3)]
    e_ids = [cid for cid, _ in search_emb(q["question"], 3)]
    a_llm, pt_llm, _ = ask([{"role": "user", "content": q["question"]}])            # system 1: no context
    a_bm,  pt_bm,  _ = ask(rag_messages(q["question"], b_ids))                        # system 2: BM25 top 3
    a_em,  pt_em,  _ = ask(rag_messages(q["question"], e_ids))                        # system 3: embedding top 3
    runs[qid] = {"llm": a_llm, "bm25": a_bm, "emb": a_em, "bm25_ids": b_ids, "emb_ids": e_ids,
                 "pt": {"llm": pt_llm, "bm25": pt_bm, "emb": pt_em}}
    print("=" * 100)
    print(qid, "|", q["question"])
    print("REFERENCE:", q["reference"])
    print("relevant chunks:", q["relevant"])
    print(f"\n[LLM only]   (prompt_tokens {pt_llm})\n{a_llm}")
    print(f"\n[BM25 RAG]   chunks {b_ids}  (prompt_tokens {pt_bm})\n{a_bm}")
    print(f"\n[Embedding RAG] chunks {e_ids}  (prompt_tokens {pt_em})\n{a_em}")

C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
relevant chunks: ['GO-04', 'GO-15']

[LLM only]   (prompt_tokens 89)
**What you’ll get**

In England the “income‑band 2” category is for households whose **adjusted net income is between £25,001 and £30,000** (the exact figure you report on the application determines where you fall inside the band).

| Household income | Maintenance‑loan amount (2023‑24 rates) |
|------------------|------------------------------------------|
| £25,001 – £26,000 | £9 500 |
| £26,001 – £27,000 | £9 250 |
| £27,001 – £28,000 | £9 000 |
| £28,001 – £29,000 | £8 750 |
| £29,001 – £30,000 | £8 500 |

So, a student whose household income puts them in **income‑band 2** will receive **between £8 500 and £9 500** for the academic year, depending on the exact income figure you give.

**When it’s paid**

The loan is paid **directly to your ban

The answers above are the evidence; the table below marks each one against `reference`.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | | | |
| C-02 | | | |
| C-03 | | | |
| C-04 | | | |
| C-05 | | | |
| C-06 | | | |
| **agrees with `reference`** | /6 | /6 | /6 |

## C3 — question 1 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [11]:
def show_case(qid):
    q = next(x for x in qa if x["id"] == qid); r = runs[qid]
    print(qid, "|", q["question"]); print("REFERENCE:", q["reference"]); print("relevant:", q["relevant"], "\n")
    print("1. BM25 top 3:       ", [(c, round(s, 3)) for c, s in search_bm25(q["question"], 3)])
    print("2. EmbeddingGemma top 3:", [(c, round(s, 3)) for c, s in search_emb(q["question"], 3)])
    for name, key in (("BM25", "bm25_ids"), ("EmbeddingGemma", "emb_ids")):
        print(f"\n3. Context sent to the model by {name} RAG:")
        print(context_block(r[key]))
    print("\nAnswers:")
    for name, key in (("LLM only", "llm"), ("BM25 RAG", "bm25"), ("Embedding RAG", "emb")):
        print(f"- {name}: {r[key]}")

show_case("C-03")

C-03 | What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
relevant: ['GO-02'] 

1. BM25 top 3:        [('GO-16', 15.799), ('GO-02', 14.059), ('GO-06', 13.852)]
2. EmbeddingGemma top 3: [('GO-02', 0.629), ('GO-20', 0.569), ('GO-05', 0.545)]

3. Context sent to the model by BM25 RAG:
[1] (GO-16 | Combining the grant with other support)
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] (GO-02 | Who qualifies)
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0 scale; their hous

## C3 — question 2 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [12]:
show_case("C-06")

C-06 | I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
relevant: ['GO-20'] 

1. BM25 top 3:        [('GO-16', 16.996), ('GO-01', 14.173), ('GO-20', 13.72)]
2. EmbeddingGemma top 3: [('GO-20', 0.666), ('GO-16', 0.616), ('GO-01', 0.554)]

3. Context sent to the model by BM25 RAG:
[1] (GO-16 | Combining the grant with other support)
The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] (GO-01 | About the need-based study grant 2026)
The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low

## C3 — question 3 of 3: (id)

**1. What BM25 retrieved** (top 3, ids and scores):

>

**2. What EmbeddingGemma retrieved** (top 3, ids and scores):

>

**3. What was sent to the model** (the context blocks):

>

**4. How retrieval changed the answer:** more accurate, more specific, more grounded, or wrong because the wrong chunks came back?

>

**5. What was missed, and was it a retrieval problem or a generation problem?** Quote the evidence.

>

In [13]:
show_case("C-01")

C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.
relevant: ['GO-04', 'GO-15'] 

1. BM25 top 3:        [('GO-04', 20.219), ('GO-02', 18.742), ('GO-01', 17.243)]
2. EmbeddingGemma top 3: [('GO-04', 0.585), ('GO-03', 0.484), ('GO-15', 0.451)]

3. Context sent to the model by BM25 RAG:
[1] (GO-04 | How much the grant pays)
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] (GO-02 | Who qualifies)
An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.

---
# Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

>

**2. When can BM25 perform better than embeddings?**

>

**3. When can embeddings perform better than BM25?**

>

**4. Why is cosine similarity used with embeddings?**

>

**5. What information is actually sent to an LLM in a RAG system?**

>

**6. Does the LLM search the original documents itself?**

>

**7. What happens if retrieval returns an irrelevant chunk?**

>

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse?** Report K = 1, 3 and 8 on one question.

>

In [14]:
# K = 1, 3, 8 with EmbeddingGemma on two questions
for qid in ("C-03", "C-01"):
    q = next(x for x in qa if x["id"] == qid)
    print("=" * 100); print(qid, "|", q["question"]); print("REFERENCE:", q["reference"])
    for K in (1, 3, 8):
        cids = [c for c, _ in search_emb(q["question"], K)]
        a, pt, _ = ask(rag_messages(q["question"], cids))
        print(f"\n--- K = {K}: chunks {cids}, relevant found: {[c for c in cids if c in q['relevant']]}, prompt_tokens {pt}")
        print(a)

C-03 | What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

--- K = 1: chunks ['GO-02'], relevant found: ['GO-02'], prompt_tokens 249
The minimum GPA required is **2.67 on the 4.0 scale**【1】.

--- K = 3: chunks ['GO-02', 'GO-20', 'GO-05'], relevant found: ['GO-02'], prompt_tokens 425
The minimum GPA required to qualify for the need‑based grant is **2.67** on the 4.0 scale【1】.

--- K = 8: chunks ['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17'], relevant found: ['GO-02'], prompt_tokens 920
The minimum cumulative GPA required to qualify for the need‑based grant is **2.67 on the 4.0 scale**【1】.
C-01 | How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

--- K = 1: chunks ['GO-04'], relevant found: ['GO-04'], prompt_tokens 244
A st

**9. Is a vector database required to build a RAG system?**

>

**10. Which retrieval approach worked better for this corpus, and what evidence supports that?** Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.

>

---
# Conclusion

>